# GPU benchmark — TokenVector.Inference vs PyTorch (Colab T4)

GPT-2 124M, seq1024, fwd+bwd — **một protocol duy nhất**:
cùng VM, cùng input `[(i*37+11)%50257]`, seed 7, 2 thread, **xen kẽ A/B 3 vòng mỗi bên, median**.

| Cell | Nội dung |
|---|---|
| [0/5] | nvidia-smi |
| [2/5] | .NET 10 SDK install |
| [3/5] | Clone/pull 2 repos + cài torch CUDA wheel |
| [4/5] | Build `.so` native (AVX2 + tkv_prof.c + tkv_attn_cuda.c) |
| [4.5/5] | Build `tkv_torch_fused.so` (bind ATen; TKV_TORCH_GPU=1 → CUDA) |
| [5/5] | `.tkv` native vs torch CPU 2T (fair) |
| [6/5] | `.tkv` GPU vs torch CUDA (fair) |
| [7/5] | TÁCH OP (TKV_PROF=1 — diagnostic, không phải bench) |
| [8/5] | Peak RAM (.tkv vs torch) |

> Loss `.tkv` ≠ torch (khởi tạo khác; FLOP giống) — chỉ so **time**.
> Đúng đắn kiểm bằng fixtures/parity `tv/tests/`, không phải bench.


In [ ]:
# GPU benchmark — TokenVector.Inference vs PyTorch (Colab T4)

GPT-2 124M, seq1024, fwd+bwd — **một protocol doc lap**:
cùng VM, cùng input `[(i*37+11)%50257]`, seed 7, 2 thread, **xen kẽ A/B 3 vòng mỗi bên, median**.

**GPU-only** (không thêm bench CPU):

- Cell [0/5]: nvidia-smi
- Cell [2/5]: cài .NET 10 SDK
- Cell [3/5]: clone (hoặc pull) repos + cài torch CUDA
- Cell [4/5]: build `.so` native (AVX2 + tkv_prof.c + tkv_attn_cuda.c)
- Cell [4.5/5]: build `tkv_torch_fused.so` (bind ATen; TKV_TORCH_GPU=1 → CUDA)
- Cell [5/5]: `.tkv` GPU (ATen bridge) vs torch CUDA — fair
- Cell [7] / tách op (TKV_PROF=1, diagnostic)
- Cell [8] / peak RAM (native CPU .tkv — chỉ tham chiếu)

> Loss `.tkv` ≠ torch (FLOPs giống, init scheme khác) — chỉ so **thời gian**.
> Đúng đắn kiểm bằng fixtures/parity `tv/tests/`, không phải bench.


In [ ]:
# [2/5] Install .NET 10 SDK
!apt-get update -qq && apt-get install -y -qq build-essential gcc libicu74 libssl3 libkrb5-3 zlib1g >/dev/null 2>&1 || apt-get install -y -qq build-essential gcc libicu-dev libkrb5-dev >/dev/null
!for u in https://dot.net/v1/dotnet-install.sh https://builds.dotnet.microsoft.com/dotnet/scripts/v1/dotnet-install.sh; do     curl -fsSL --retry 5 --retry-delay 3 --connect-timeout 20 "$u" -o /tmp/dotnet-install.sh && break;     echo "that bai: $u"; done; ls -l /tmp/dotnet-install.sh
!bash /tmp/dotnet-install.sh --channel 10.0 --install-dir /opt/dotnet
import os
os.environ["PATH"] = "/opt/dotnet:" + os.environ["PATH"]
!/opt/dotnet/dotnet --info | head -15


In [ ]:
# [3/5] Clone (or update) repos + torch CUDA + LD_LIBRARY_PATH
!git clone --depth 1 https://github.com/nguyenhungtran18/TokenVector.git || (cd TokenVector && git pull --depth 1)
!git clone --depth 1 https://github.com/nguyenhungtran18/TokenVector.Inference.git || (cd TokenVector.Inference && git pull --depth 1)
import os, glob
cublas_dir = next(iter(glob.glob('/usr/local/lib/python*/dist-packages/nvidia/cublas/lib')), None)
rt_dir = next(iter(glob.glob('/usr/local/lib/python*/dist-packages/nvidia/cuda_runtime/lib')), None)
torch_dir = next(iter(glob.glob('/usr/local/lib/python*/dist-packages/torch')), None)
print('cublas_dir:', cublas_dir)
print('rt_dir     :', rt_dir)
print('torch      :', torch_dir)
env_ld = cublas_dir or ''
if rt_dir: env_ld = env_ld + ':' + rt_dir
if torch_dir: env_ld = env_ld + ':' + torch_dir + '/lib'
if env_ld: os.environ['LD_LIBRARY_PATH'] = env_ld + (':' + os.environ['LD_LIBRARY_PATH'] if os.environ.get('LD_LIBRARY_PATH') else '')
print('LD_LIBRARY_PATH =', os.environ.get('LD_LIBRARY_PATH'))


In [ ]:
# [4/5] Build tkv_simd_native.so (AVX2 + TKV_PROF + cuBLAS dispatch)
!cd TokenVector/Release/tools/simd_native && gcc -O2 -mavx2 -mfma -shared -fPIC \
  tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_prof.c tkv_attn_cuda.c \
  -lm -lpthread -ldl -o tkv_simd_native.dll
!cp TokenVector/Release/tools/simd_native/tkv_simd_native.dll TokenVector.Inference/demo/tkv_simd_native.dll
!cp TokenVector/Release/tools/simd_native/tkv_simd_native.dll TokenVector.Inference/tkv_simd_native.dll
!cd TokenVector/Release/tools/simd_native && gcc -O2 -mavx2 -mfma -shared -fPIC tkv_step124.c -o tkv_step124.dll
!cp TokenVector/Release/tools/simd_native/tkv_step124.dll TokenVector.Inference/demo/tkv_step124.dll
print('Built .so')
import ctypes
lib = ctypes.CDLL('/content/TokenVector.Inference/demo/tkv_simd_native.dll')
lib.TkvCUDA_Available.restype = ctypes.c_int
print('TkvCUDA_Available() =', lib.TkvCUDA_Available(), '(1 = cuBLAS on GPU, 0 = CPU fallback)')


In [ ]:
# [4.5/5] Build tkv_torch_fused.so (bind ATen — TKV_TORCH_GPU=1 → CUDA)
import subprocess, os, sys
torch_dir = subprocess.check_output([sys.executable, '-c', 'import torch,os; print(os.path.dirname(torch.__file__))']).decode().strip()
print('torch:', torch_dir)
inc1 = torch_dir + '/include'
inc2 = torch_dir + '/include/torch/csrc/api/include'
libdir = torch_dir + '/lib'
cmd = ('cd /content/TokenVector/Release/tools/simd_native && '
      + 'g++ -O2 -shared -std=c++20 -fPIC torch_fused.cpp -o tkv_torch_fused.so '
      + '-I' + inc1 + ' -I' + inc2 + ' -L' + libdir + ' '
      + '-Wl,-rpath,' + libdir + ' '
      + '-ltorch -ltorch_cpu -lc10 -lc10_cuda -ltorch_cuda -Wl,-rpath-link,' + libdir + ' '
      + '2>&1 | tail -40')
print(subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout)
so_path = '/content/TokenVector/Release/tools/simd_native/tkv_torch_fused.so'
print('exists:', os.path.exists(so_path))
os.environ['TKV_TORCH_PATH'] = so_path
os.environ['LD_LIBRARY_PATH'] = libdir + ':' + os.environ.get('LD_LIBRARY_PATH', '')
import ctypes
lib = ctypes.CDLL(so_path)
print('ctypes load OK (tkv_torch_fused)')


In [ ]:
# [6/5] GPU: .tkv bridge GPU vs torch CUDA — xen ke x3, median
import subprocess, re, statistics as st
DOT = '$(command -v dotnet || echo /opt/dotnet/dotnet)'
def run_cmd(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=1800).stdout
def tkv_gpu():
    out = run_cmd('cd TokenVector.Inference && printf "1024\n2\n1\n" > tv/tests/_seq_cfg.txt && '
                  + 'TKV_TORCH_GPU=1 ' + DOT + ' demo/bseq.dll')
    m = re.search(r'fwd=([\d.]+) loss=([\d.]+) bwd=([\d.]+) opt=([\d.]+) copyback=([\d.]+) TONG=([\d.]+)', out)
    return dict(zip(('fwd','ce','bwd','opt','copyback','total'), map(float, m.groups())))
def torch_cuda():
    out = run_cmd('cd TokenVector.Inference && TKV_BENCH_DEVICE=cuda python3 benchmark/bench_torch.py d 1024')
    def g(pat):
        m = re.search(pat + r'[^\n]*?median=([\d.]+)', out)
        return float(m.group(1))
    return dict(fwdbwd=g(r'fwd\+bwd seq1024'), full=g(r'full step seq1024'))
tk, tt = [], []
for r in range(3):
    if r % 2 == 0:
        a, b = tkv_gpu(), torch_cuda()
    else:
        b, a = torch_cuda(), tkv_gpu()
    tk.append(a); tt.append(b)
    print('vong %d: .tkv fwd=%.0f CE=%.0f bwd=%.0f opt=%.0f TONG=%.0f | torch fwd+bwd=%.1f full=%.1f ms' % (
        r+1, a['fwd'], a['ce'], a['bwd'], a['opt'], a['total'], b['fwdbwd'], b['full']))
med = st.median
fb  = med([x['fwd']+x['ce']+x['bwd'] for x in tk])
tot = med([x['total'] for x in tk])
tfb = med([x['fwdbwd'] for x in tt])
ttot = med([x['full'] for x in tt])
print('\n==== MEDIAN (n=3, xen ke) ====')
print('.tkv GPU fwd+CE+bwd = %.1f ms | torch fwd+bwd = %.1f ms | ty le = %.2fx' % (fb, tfb, fb/tfb))
print('.tkv GPU TONG (full) = %.1f ms | torch full step = %.1f ms | ty le = %.2fx' % (tot, ttot, tot/ttot))


In [ ]:
# [7/5] TACH OP (diagnostic, KHONG phai bench) — native vs GPU bridge
DOT = '$(command -v dotnet || echo /opt/dotnet/dotnet)'
def run(cmd):
    import subprocess
    p   = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=1800)
    out = (p.stdout or '') + '\n' + (p.stderr or '')
    for line in out.splitlines():
        if 'BPHASE' in line or 'TKV_PROF' in line:
            print(line.rstrip())
print('=== native (TKV_TORCH_PATH=) ===')
run('cd TokenVector.Inference && printf "1024\n2\n1\n" > tv/tests/_seq_cfg.txt && TKV_TORCH_PATH= TKV_PROF=1 ' + DOT + ' demo/bseq.dll')
print('=== GPU bridge (TKV_TORCH_GPU=1) ===')
run('cd TokenVector.Inference && printf "1024\n2\n1\n" > tv/tests/_seq_cfg.txt && TKV_TORCH_GPU=1 TKV_PROF=1 ' + DOT + ' demo/bseq.dll')


In [ ]:
# [extra] DIAGNOSTIC — vi sao ATen attention backward khong chay (hoac chay)
# Chay day du GPU bridge + TKV_PROF=1 va in stderr [tkv-attn-bwd] + BPHASE
DOTCMD = 'cd TokenVector.Inference && printf "1024\n2\n1\n" > tv/tests/_seq_cfg.txt && '
!cd TokenVector.Inference && printf "1024\n2\n1\n" > tv/tests/_seq_cfg.txt && TKV_TORCH_GPU=1 TKV_PROF=1 $(command -v dotnet || echo /opt/dotnet/dotnet) demo/bseq.dll 2>&1 | grep -E 'BPHASE|tkv-attn-bwd|FAIL'


In [ ]:
# [8/5] Peak RAM (VmHWM)
!apt-get install -y -qq time >/dev/null 2>&1 || true
print('--- .tkv native, mode 0 seq1024, VmHWM ---')
!cd TokenVector.Inference && printf '1024\n0\n1\n' > tv/tests/_seq_cfg.txt && TKV_TORCH_PATH= /usr/bin/time -v $(command -v dotnet || echo /opt/dotnet/dotnet) demo/bseq.dll 2>&1 | grep -E 'BSEQ|Maximum resident'
print('--- PyTorch CPU (tier_d = fwd+bwd, T=2) ---')
!cd TokenVector.Inference && TKV_BENCH_DEVICE=cpu TKV_BENCH_THREADS=2 /usr/bin/time -v python3 benchmark/bench_torch.py d 1024 2>&1 | grep -E 'TOTAL|Maximum resident'
